In [3]:
# setup and load
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.utils import get_engine

engine = get_engine()

names = ["orders", "customers", "order_items", "order_payments",
         "order_reviews", "products", "sellers", "category_translation"]
tables = {n: pd.read_sql(f"SELECT * FROM {n}", engine) for n in names}

orders      = tables["orders"]
customers   = tables["customers"]
items       = tables["order_items"]
payments    = tables["order_payments"]
reviews     = tables["order_reviews"]
products    = tables["products"]
translation = tables["category_translation"]

for n, df in tables.items():
    print(f"{n:22s} {df.shape}")

orders                 (99441, 8)
customers              (99441, 5)
order_items            (112650, 7)
order_payments         (103886, 5)
order_reviews          (99224, 7)
products               (32951, 9)
sellers                (3095, 4)
category_translation   (71, 2)


In [4]:
# Missing Values
for n, df in tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f"\n{n}")
        print(nulls.to_string())


orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

order_reviews
review_comment_title      87656
review_comment_message    58247

products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2


In [5]:
# Duplicates
print("orders duplicate order_id:     ", orders["order_id"].duplicated().sum())
print("customers duplicate customer_id:", customers["customer_id"].duplicated().sum())
print("items duplicate (order,item):   ", items.duplicated(["order_id", "order_item_id"]).sum())
print("orders with >1 review:          ", reviews["order_id"].duplicated().sum())
print("customer_ids vs real people:    ", customers["customer_id"].nunique(),
      "vs", customers["customer_unique_id"].nunique())

orders duplicate order_id:      0
customers duplicate customer_id: 0
items duplicate (order,item):    0
orders with >1 review:           551
customer_ids vs real people:     99441 vs 96096


In [6]:
# Data Sanity Check
d = orders[orders["order_status"] == "delivered"].copy()

print("delivered but no delivery date:", d["order_delivered_customer_date"].isna().sum())

bad = d[d["order_delivered_customer_date"] < d["order_purchase_timestamp"]]
print("delivered before purchase:     ", len(bad))

print("date range:", d["order_purchase_timestamp"].min(), "to", d["order_purchase_timestamp"].max())

delivered but no delivery date: 8
delivered before purchase:      0
date range: 2016-09-15 12:16:38 to 2018-08-29 15:00:37


In [7]:
# Delivery Features
d = d[d["order_delivered_customer_date"].notna()].copy()
d = d[d["order_delivered_customer_date"] >= d["order_purchase_timestamp"]]

d["delivery_days"] = (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.days
d["delay_days"]    = (d["order_delivered_customer_date"] - d["order_estimated_delivery_date"]).dt.days
d["is_late"]       = (d["delay_days"] > 0).astype(int)
d["purchase_month"] = d["order_purchase_timestamp"].dt.to_period("M").astype(str)

print(d[["delivery_days", "delay_days"]].describe().round(1))
print("late share:", d["is_late"].mean().round(3))

       delivery_days  delay_days
count        96470.0     96470.0
mean            12.1       -11.9
std              9.6        10.2
min              0.0      -147.0
25%              6.0       -17.0
50%             10.0       -12.0
75%             15.0        -7.0
max            209.0       188.0
late share: 0.068


In [8]:
# One review per Order
reviews_one = (reviews.sort_values("review_answer_timestamp")
                      .drop_duplicates("order_id", keep="last")
                      [["order_id", "review_score"]])
print(len(reviews), "->", len(reviews_one))

99224 -> 98673


In [9]:
# Aggregate items before joining
item_totals = (items.groupby("order_id")
                    .agg(items_count=("order_item_id", "count"),
                         items_value=("price", "sum"),
                         freight_value=("freight_value", "sum"))
                    .reset_index())
item_totals["order_value"] = item_totals["items_value"] + item_totals["freight_value"]

In [10]:
# Build the Clean Table
cust = customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]]

before = len(d)
orders_clean = (d.merge(cust, on="customer_id", how="left")
                 .merge(item_totals, on="order_id", how="left")
                 .merge(reviews_one, on="order_id", how="left"))

assert len(orders_clean) == before, "a join duplicated rows"
assert orders_clean["order_id"].is_unique

print(orders_clean.shape)
print(orders_clean.isna().sum()[lambda s: s > 0])

(96470, 20)
order_approved_at                14
order_delivered_carrier_date      1
review_score                    646
dtype: int64


In [11]:
# Clean Product Categories
products_clean = (products.merge(translation, on="product_category_name", how="left"))
products_clean["category"] = (products_clean["product_category_name_english"]
                              .fillna("unknown"))
print(products_clean["category"].value_counts().head(10))

category
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


In [12]:
# Save the OutPut
orders_clean.to_csv("../data/processed/orders_clean.csv", index=False)
orders_clean.to_sql("orders_clean", engine, if_exists="replace", index=False, chunksize=5000)
print("saved", len(orders_clean), "rows")

saved 96470 rows
